=================================================
### Práctica 4 — Detección y Seguimiento con YOLOv8
=================================================

##### Este cuaderno realiza:
1. Detección y seguimiento de personas y vehículos con **YOLOv8**.
2. Detección de matrículas con un modelo propio entrenado.
3. Anonimización facial de peatones y de matrículas.
4. Generación de un **CSV con detecciones y tracking**.
5. Cálculo posterior del **flujo vertical (arriba / abajo)** y conteo por clases.


### Carga de modelos y configuración inicial

In [ ]:
from ultralytics import YOLO
import cv2, csv
from collections import defaultdict, deque
import pandas as pd

# Archivos de entrada y salida
video_input = "C0142.mp4"
video_output = "p4_output.mp4"
csv_output   = "p4_results.csv"

# Modelos YOLO
general_model = YOLO('yolo11n.pt').to('cuda')  # Detección general (GPU)
plate_model   = YOLO('yolo_runs/plates_detection/weights/best.pt').to('cuda')  # Detección de matrículas (GPU)

# Clases relevantes
classNames = ["person", "bicycle", "car", "motorbike", "bus", "truck"]
general_classes = [0,1,2,3,5,7]  # IDs de COCO para personas y vehículos comunes

### Preparación del vídeo

In [ ]:
# Carga del vídeo
vid = cv2.VideoCapture(video_input)
width  = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = vid.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter(video_output, fourcc, fps, (width, height))

# Historial de tracking
track_history = defaultdict(lambda: deque(maxlen=5))

### Funciones auxiliares para el CSV

In [ ]:
def write_csv_header(path):
    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["frame","tipo_objeto","confianza","track_id","x1","y1","x2","y2",
                         "plate","plate_conf","mx1","my1","mx2","my2"])

def append_detection(path, frame, obj_class, conf, track_id,
                     x1, y1, x2, y2, plate, plate_conf, mx1, my1, mx2, my2):
    with open(path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([frame,obj_class,conf,track_id,x1,y1,x2,y2,
                         plate,plate_conf,mx1,my1,mx2,my2])

write_csv_header(csv_output)

### Procesamiento frame a frame con detección, seguimiento y anonimización

In [ ]:
frame_n = 0

while True:
    ret, frame = vid.read()
    if not ret:
        break

    frame_n += 1

    results = general_model.track(frame, persist=True, classes=general_classes, device='cuda')

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls >= len(classNames):
                continue
            clase = classNames[cls]
            conf = float(box.conf[0])
            track_id = int(box.id[0]) if box.id is not None else -1
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Anonimizar rostro (solo en personas)
            if clase == "person":
                face_h = int((y2 - y1) * 0.2)
                face_region = frame[y1:y1+face_h, x1:x2]
                if face_region.size > 0:
                    small = cv2.resize(face_region, (0,0), fx=0.1, fy=0.1)
                    pixel = cv2.resize(small, (face_region.shape[1], face_region.shape[0]), interpolation=cv2.INTER_NEAREST)
                    frame[y1:y1+face_h, x1:x2] = pixel

            # Detección de matrícula
            plate_conf, mx1, my1, mx2, my2 = "", "", "", "", ""
            has_plate = False

            if clase != "person":
                vehicle_crop = frame[y1:y2, x1:x2]
                results_plate = plate_model(vehicle_crop, device='cuda')
                for p in results_plate:
                    for pb in p.boxes:
                        conf_plate = float(pb.conf[0])
                        if conf_plate > 0.25:
                            has_plate = True
                            x1p, y1p, x2p, y2p = map(int, pb.xyxy[0])
                            plate_conf = conf_plate
                            mx1, my1, mx2, my2 = x1p, y1p, x2p, y2p

                            # Pixelar matrícula
                            plate_region = vehicle_crop[y1p:y2p, x1p:x2p]
                            if plate_region.size > 0:
                                small = cv2.resize(plate_region, (0,0), fx=0.1, fy=0.1)
                                pixel = cv2.resize(small, (plate_region.shape[1], plate_region.shape[0]), interpolation=cv2.INTER_NEAREST)
                                vehicle_crop[y1p:y2p, x1p:x2p] = pixel

            # Dibujar y guardar CSV
            color = (0,255,0) if clase != "person" else (255,255,0)
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, f"{track_id}-{clase} ({conf:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            append_detection(csv_output, frame_n, clase, conf, track_id,
                             x1, y1, x2, y2, 
                             "Si" if has_plate else "No", plate_conf, mx1, my1, mx2, my2)

    out_vid.write(frame)

vid.release()
out_vid.release()
print("CSV de detecciones generado:", csv_output)

### Conteo de clases

In [ ]:
# Leer el CSV con detecciones
df = pd.read_csv("p4_results.csv")

# Asegurar que el track_id sea string (evita errores de mezcla int/str)
df["track_id"] = df["track_id"].astype(str)

# 1️⃣ Eliminar duplicados por track_id → 1 fila por objeto único
df_unique = df.drop_duplicates(subset="track_id", keep="last")

# 2️⃣ Conteo por tipo de objeto
conteo_clases = df_unique["tipo_objeto"].value_counts().reset_index()
conteo_clases.columns = ["tipo_objeto", "conteo"]

# 3️⃣ Guardar en CSV
conteo_clases.to_csv("p4_conteo_clases.csv", index=False)

print("✅ Conteo de clases sin repeticiones guardado en 'p4_conteo_clases.csv'")
display(conteo_clases)


### Cálculo de flujo vertical

In [ ]:
input_csv = "p4_results.csv"
output_csv = "p4_flujo.csv"

track_history = defaultdict(list)
track_last_row = {}

# Registrar el centro vertical (cy)
with open(input_csv, newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    for row in rows:
        track_id = row["track_id"]
        y1, y2 = int(row["y1"]), int(row["y2"])
        cy = (y1 + y2) // 2
        track_history[track_id].append(cy)
        track_last_row[track_id] = row

# Calcular flujo
flows = {}
for tid, cy_list in track_history.items():
    if len(cy_list) < 2:
        continue
    y_in, y_out = cy_list[0], cy_list[-1]
    if y_out < y_in:
        flows[tid] = "arriba"
    elif y_out > y_in:
        flows[tid] = "abajo"
    else:
        flows[tid] = "estático"

# Guardar CSV final (una fila por track_id)
fieldnames = list(rows[0].keys()) + ["flujo"]
with open(output_csv, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for tid, flow in flows.items():
        row = track_last_row[tid]
        row["flujo"] = flow
        writer.writerow(row)

print("Flujo vertical calculado y guardado en:", output_csv)

### Análisis de flujo

In [ ]:
# Leer el CSV con flujo ya calculado
df = pd.read_csv("p4_flujo.csv")

# Asegurar que el track_id sea string para evitar errores
df["track_id"] = df["track_id"].astype(str)

# Eliminar duplicados por track_id (quedarnos con la última aparición de cada objeto)
df_unique = df.drop_duplicates(subset="track_id", keep="last")

# Conteo de flujo vertical (por dirección)
conteo_flujo = df_unique.groupby(["tipo_objeto", "flujo"]).size().reset_index(name="cantidad")

# Guardar ambos resultados
conteo_flujo.to_csv("p4_conteo_flujo.csv", index=False)

print(" - Conteo por flujo → p4_conteo_flujo.csv")

print("\nFlujo vertical (por clase y dirección):")
display(conteo_flujo)